## About this notebook — `algorithms_explorer.ipynb`

**Purpose:** Complete reference and sandbox for every processing function in the
`fingerprints/` package. Each of the 12 phases is self-contained — jump to any phase,
call any function, and compare results side by side using the `show()` helper.

| | |
|---|---|
| **Input** | Any fingerprint image (photo or scanned); change `IMAGE_PATH` at top |
| **Output** | Side-by-side matplotlib comparisons; no files written except in Phase 10 |
| **Pipeline** | Static matplotlib cells — not widget-interactive; re-run cells individually |
| **Best for** | Learning what each function does; choosing algorithms before building a pipeline |

### Phases covered

| Phase | Functions |
|:---:|:---|
| 0 — Geometry | `cylindrical_unwrap` · `rectify_perspective` |
| 1 — Grayscale | All 7 channel variants |
| 2 — Masking | `compute_fingerprint_roi` · `mask_from_black_background` · `build_foreground_mask` |
| 3 — Illumination | `homomorphic_filter` · `subtract_rolling_background` · `subtract_heavy_median` · `local_normalize` |
| 4a — Contrast | `apply_clahe` · `gamma_correction` · `local_contrast_enhancement` · `contrast_stretch_saturated` |
| 4b — Frequency | `difference_of_gaussians` · `fft_gaussian_bandpass` |
| 4c — Ridge-aware | `apply_ridge_filter` · `coherence_enhancing_diffusion` · `gabor_rebuild` · `dynamic_gabor_pipeline` |
| 5 — Binarization | 10 methods: fixed · otsu · li · triangle · mean · adaptive · sauvola · niblack … |
| 6 — Morphology | All 9 operations |
| 7 — Skeleton | skimage vs OpenCV · spur pruning · component removal |
| 8 — Minutiae | Extraction · finalization · visualisation |
| 9 — Quality & Singularities | `estimate_image_quality` · orientation map · core/delta detection |
| 10 — Scale & DPI | `ScaleCalibrator` · `rescale_to_print_dpi` · `normalize_to_dpi` |
| 11 — Matching | `compare_fingerprints` · match score |

---

### Notebook comparison

| | Notebook | Input | Output | Pipeline | Best for |
|---|:---|:---|:---|:---|:---|
| | scanning | Live scanner | Captured images | Buttons | Acquire fingerprints from hardware |
| | scanned_small_pipeline | Scanned image | Skeleton + minutiae | Widget sliders | Quick processing, sensible defaults |
| | scanned_full_pipeline | Scanned image | Skeleton + minutiae | Widget sliders + toggles | Full parameter control, scanned FP |
| | photo_finger | Photo (dark background) | Skeleton + minutiae | Widget sliders | Full photo pipeline (ImageJ-based) |
| | ridge_valley | Photo or scanned | Valley print map | Widgets + polygon ROI | Valley extraction with manual ROI |
| | photo_valley_pipeline | Photo | Valley print map | Static comparisons | Experiment with photo → valley methods |
| ▶ | **algorithms_explorer** | Any | Side-by-side comparisons | Static comparisons | Learn and compare all available algorithms |


# Fingerprint Algorithm Explorer

All processing functions in the `fingerprints/` package, organised by pipeline phase.
Each section is self-contained — run the **Setup** and **Load image** cells first, then jump to any phase.

| Phase | What it does |
|---|---|
| 0 — Geometry | Perspective / cylindrical unwarp |
| 1 — Grayscale | Channel selection for colour photos |
| 2 — Masking | ROI segmentation |
| 3 — Illumination | Background & uneven-light removal |
| 4 — Enhancement | Contrast, edge, ridge amplification |
| 5 — Binarization | Ridge/valley separation |
| 6 — Morphology | Binary cleanup |
| 7 — Skeletonisation | Ridge thinning |
| 8 — Minutiae | Feature point extraction |
| 9 — Quality & Singularities | Image quality score, core/delta detection |
| 10 — Scale & DPI | Physical measurement, print rescaling |
| 11 — Matching | Compare two fingerprints |


## Setup

In [ ]:
import sys, importlib
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import ipywidgets as widgets
from IPython.display import display

import fingerprints.gray        as fpgray
import fingerprints.mask        as fpmask
import fingerprints.enhancement as fpenh
import fingerprints.binarization as fpbin
import fingerprints.morphology  as fpmorph
import fingerprints.skeleton    as fpskel
import fingerprints.minutiae    as fpmin
import fingerprints.filters     as fpfilt
import fingerprints.frequency   as fpfreq
import fingerprints.singularity as fpsin
import fingerprints.geometry    as fpgeom
import fingerprints.matching    as fpmat
import fingerprints.quality     as fpqual
import fingerprints.utils       as fputils
try:
    import fingerprints.calibration as fpcal
except Exception as e:
    fpcal = None
    print(f"calibration module unavailable: {e}")

for mod in [fpgray, fpmask, fpenh, fpbin, fpmorph, fpskel, fpmin,
            fpfilt, fpfreq, fpsin, fpgeom, fpmat, fpqual, fputils]:
    importlib.reload(mod)

normalize = fputils.normalize_gray
print("All modules loaded. PROJECT_ROOT:", PROJECT_ROOT)


## Load image
Change `IMAGE_PATH` to point at any fingerprint file.

In [ ]:
IMAGE_PATH = PROJECT_ROOT / "images" / "photo" / "image4.jpeg"

img_bgr  = cv2.imread(str(IMAGE_PATH))
img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY) if img_bgr is not None else None
if img_gray is None:
    raise RuntimeError(f"Could not load {IMAGE_PATH}")

print(f"Loaded {IMAGE_PATH}  shape={img_gray.shape}  dtype={img_gray.dtype}")
plt.figure(figsize=(4, 5))
plt.imshow(img_gray, cmap="gray"); plt.title("Input"); plt.axis("off"); plt.show()


## Visualisation helper
`show(items)` — pass a list of `(title, image)` tuples.

In [ ]:
def show(items, cols=4, figsize_per=3, cmap="gray"):
    """Display a list of (title, ndarray) side by side."""
    n = len(items)
    cols = min(cols, n)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols,
                             figsize=(figsize_per * cols, figsize_per * rows))
    axes = np.array(axes).reshape(-1)
    for ax, (title, img) in zip(axes, items):
        if img is None:
            ax.set_visible(False); continue
        if img.ndim == 3:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        else:
            ax.imshow(img, cmap=cmap, vmin=0, vmax=255)
        ax.set_title(title, fontsize=9)
        ax.axis("off")
    for ax in axes[n:]:
        ax.set_visible(False)
    plt.tight_layout(); plt.show()

print("show() helper ready.")


---
## Phase 0 — Geometric correction

Correct lens distortion or finger curvature **before** any processing.

| Function | When to use |
|---|---|
| `rectify_perspective(img, src_pts)` | Photographed at an angle — supply 4 corner points |
| `cylindrical_unwrap(img, focal_length)` | Finger pressed on curved surface, edges compressed |


In [ ]:
# ── Cylindrical unwrap ─────────────────────────────────────────────────────
# focal_length: larger → less correction; typical range 400–1200
focal_length = 800

unwrapped = fpgeom.cylindrical_unwrap(img_gray, focal_length=focal_length)

show([
    ("Original", img_gray),
    (f"Cylindrical unwrap  f={focal_length}", unwrapped),
])

# ── Perspective rectification ──────────────────────────────────────────────
# Provide 4 (x, y) corners of the fingerprint area in the source image
# (top-left, top-right, bottom-right, bottom-left)
# Example — uncomment and edit to your image:
#
# src_pts = np.float32([[50, 30], [430, 20], [450, 580], [30, 600]])
# rectified = fpgeom.rectify_perspective(img_bgr, src_pts)
# show([("Original (BGR)", img_bgr), ("Rectified", rectified)])
print("Adjust focal_length above; uncomment the perspective block if needed.")


---
## Phase 1 — Grayscale channel selection

For **colour photos** only. Different channels can reveal ridge structure more clearly
depending on lighting and skin tone.

| Function | Formula / channel |
|---|---|
| `rgb_to_gray_luma` | 0.299R + 0.587G + 0.114B (perceptual) |
| `rgb_to_gray_green` | G channel only |
| `rgb_to_gray_red` | R channel only |
| `rgb_to_gray_blue` | B channel only |
| `rgb_to_gray_max` | max(R, G, B) |
| `rgb_to_gray_min` | min(R, G, B) |
| `rgb_to_gray_avg` | (R+G+B)/3 |


In [ ]:
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

variants = fpgray.compute_gray_variants(img_rgb)

show([(name, arr) for name, arr in variants.items()], cols=4)


---
## Phase 2 — ROI masking

Isolate the fingerprint from the background.  Mask = 255 inside, 0 outside.

| Function | Best for |
|---|---|
| `compute_fingerprint_roi` | Variance-based; good for scanned prints |
| `mask_from_black_background` | Dark (scanner) backgrounds |
| `build_foreground_mask` | Adaptive; good for photos |
| `clean_mask` | Post-process any binary mask |


In [ ]:
# ── Parameters ──────────────────────────────────────────────────────────────
# variance_roi
var_block    = 16
var_close_k  = 25
var_erode_k  = 7
var_close_it = 4
var_dilate_it= 1
var_min_area = 4000

# threshold (dark background)
thr_thresh  = 35
thr_close_k = 9

# foreground (adaptive)
fg_blur_k  = 9
fg_block   = 31
fg_C       = 7
fg_close_k = 9
fg_min_area= 4000

# ── Compute masks ────────────────────────────────────────────────────────────
mask_var = fpmask.compute_fingerprint_roi(
    img_gray, block=var_block, close_k=var_close_k,
    erode_k=var_erode_k, close_iter=var_close_it,
    dilate_iter=var_dilate_it, min_area=var_min_area)

mask_thr = fpmask.mask_from_black_background(
    img_gray, thresh=thr_thresh, close_k=thr_close_k)

mask_fg  = fpmask.build_foreground_mask(
    img_gray, blur_k=fg_blur_k, block=fg_block,
    C=fg_C, close_k=fg_close_k, min_area=fg_min_area)

def masked(img, m):
    out = img.copy(); out[m == 0] = 0; return out

show([
    ("Original",           img_gray),
    ("variance_roi mask",  mask_var),
    ("variance_roi result",masked(img_gray, mask_var)),
    ("threshold mask",     mask_thr),
    ("threshold result",   masked(img_gray, mask_thr)),
    ("foreground mask",    mask_fg),
    ("foreground result",  masked(img_gray, mask_fg)),
])

# Pick the best mask for downstream phases
MASK = mask_var   # ← change to mask_thr or mask_fg as needed
print("MASK set. Change MASK above to select a different method.")


---
## Phase 3 — Illumination & background removal

Remove uneven illumination (flash, scanner gradient) before enhancement.

| Function | Technique |
|---|---|
| `homomorphic_filter` | Frequency-domain high-pass (remove low-freq illumination) |
| `subtract_rolling_background` | ImageJ rolling-ball subtraction |
| `subtract_heavy_median` | Subtract large-kernel median (approximates background) |
| `local_normalize` | Hong-Wan-Jain block normalization |


In [ ]:
# ── Parameters ──────────────────────────────────────────────────────────────
homo_cutoff   = 30     # larger → more illumination removal
rb_radius     = 25     # rolling ball radius (pixels)
hm_ksize      = 61     # heavy median kernel size (odd)
ln_block      = 16     # local normalisation block size

# ── Compute ──────────────────────────────────────────────────────────────────
homo    = fpenh.homomorphic_filter(img_gray, cutoff=homo_cutoff)
rb      = fpenh.subtract_rolling_background(img_gray, radius=rb_radius)[0]
hm, _   = fpenh.subtract_heavy_median(img_gray, ksize=fputils.ensure_odd(hm_ksize, 1))
ln      = fpenh.local_normalize(img_gray, block_size=ln_block)

show([
    ("Original",                img_gray),
    (f"Homomorphic  cut={homo_cutoff}", homo),
    (f"Rolling ball  r={rb_radius}",    rb),
    (f"Subtract heavy median  k={hm_ksize}", hm),
    (f"Local normalize  b={ln_block}", ln),
])

# Choose one for downstream phases
ILLUM = homo   # ← change to rb / hm / ln / img_gray (no correction)
print("ILLUM set.")


---
## Phase 4a — Enhancement: basic contrast

Amplify ridge/valley contrast without structural assumptions.

| Function | Key params |
|---|---|
| `apply_clahe` | clip, tile |
| `gamma_correction` | gamma (<1 brightens, >1 darkens) |
| `local_contrast_enhancement` | radius, amount |
| `contrast_stretch_saturated` | saturated (tail %) |


In [ ]:
clahe_clip   = 2.5;  clahe_tile = 8
gamma_val    = 0.7
lce_radius   = 31;  lce_amount = 1.0
cs_saturated = 0.35

src = ILLUM   # feeds from Phase 3 output

clahe  = fpenh.apply_clahe(src, clip=clahe_clip, tile=clahe_tile)
gam    = fpenh.gamma_correction(src, gamma=gamma_val)
lce    = fpenh.local_contrast_enhancement(src, radius=lce_radius, amount=lce_amount)
cs     = fpenh.contrast_stretch_saturated(src, saturated=cs_saturated)
cm     = fpenh.clahe_median_pipeline(src, clahe_clip=clahe_clip,
                                      clahe_tile=clahe_tile, median_ksize=3)

show([
    ("Input",              src),
    (f"CLAHE  clip={clahe_clip} tile={clahe_tile}", clahe),
    (f"Gamma  γ={gamma_val}",                      gam),
    (f"Local contrast  r={lce_radius}",             lce),
    (f"Contrast stretch  sat={cs_saturated}",       cs),
    (f"CLAHE + median",                             cm),
])

ENH = clahe   # ← pick your favourite; used in Phases 4b-5
print("ENH set.")


---
## Phase 4b — Enhancement: frequency / band-pass

Emphasise the spatial frequency band corresponding to ridge spacing (~0.06–0.12 cycles/px).

| Function | Technique |
|---|---|
| `difference_of_gaussians` | sigma1 (fine detail), sigma2 (background), gain |
| `fft_gaussian_bandpass` | low_sigma (removes noise), high_sigma (removes DC) |


In [ ]:
dog_s1 = 1.0;  dog_s2 = 8.0;  dog_gain = 1.5
fft_lo = 3.0;  fft_hi  = 28.0

src = ENH

dog = fpenh.difference_of_gaussians(src, sigma1=dog_s1, sigma2=dog_s2, gain=dog_gain)
fft = fpenh.fft_gaussian_bandpass(src, low_sigma=fft_lo, high_sigma=fft_hi)

show([
    ("Input",                                    src),
    (f"DoG  σ1={dog_s1} σ2={dog_s2} g={dog_gain}", dog),
    (f"FFT bandpass  lo={fft_lo} hi={fft_hi}",     fft),
])


---
## Phase 4c — Enhancement: advanced (ridge-aware)

These methods use ridge orientation or frequency information to enhance structure selectively.

| Function | Notes |
|---|---|
| `apply_ridge_filter` | Frangi / Meijering vesselness filter |
| `coherence_enhancing_diffusion` | CED — diffuse along ridges |
| `gabor_rebuild` | Fixed-frequency Gabor bank |
| `dynamic_gabor_pipeline` | Orientation + frequency estimated first, then adaptive Gabor |


In [ ]:
# Ridge filter
rf_method    = "frangi"   # or "meijering"
rf_sigmas    = (1.0, 2.0, 3.0)
rf_black     = True       # True = dark ridges on light background

# CED
ced_lambda   = 0.1;  ced_sigma = 1.0;  ced_rho = 3.0
ced_steps    = 5;    ced_step  = 0.2

# Gabor (fixed frequency)
gab_sigma    = 3.0;  gab_freq = 0.1;  gab_angles = 12

src = ENH

rf   = fpenh.apply_ridge_filter(src, method=rf_method,
                                 sigmas=rf_sigmas, black_ridges=rf_black)
ced  = fpenh.coherence_enhancing_diffusion(src, lambda_=ced_lambda,
           sigma=ced_sigma, rho=ced_rho, step_size=ced_step, n_steps=ced_steps)
gab  = fpenh.gabor_rebuild(src, sigma=gab_sigma, freq=gab_freq,
                             n_angles=gab_angles)

# Dynamic Gabor (most sophisticated — estimates orientation & frequency per block)
try:
    dyn_enh, theta_map, freq_map = fpenh.dynamic_gabor_pipeline(src)
    dyn_label = "Dynamic Gabor"
except Exception as ex:
    dyn_enh = None; dyn_label = f"Dynamic Gabor (err: {ex})"

show([
    ("Input",                                          src),
    (f"Ridge filter  ({rf_method})",                   rf),
    (f"CED  λ={ced_lambda} steps={ced_steps}",         ced),
    (f"Gabor  σ={gab_sigma} f={gab_freq}",             gab),
    (dyn_label,                                        dyn_enh),
])

# Visualise the orientation map if dynamic Gabor worked
try:
    theta_vis = ((theta_map + np.pi/2) / np.pi * 255).astype(np.uint8)
    show([("Orientation map (angle → grey)", theta_vis),
          ("Frequency map",  (freq_map * 2000).clip(0, 255).astype(np.uint8))])
except Exception:
    pass


---
## Phase 5 — Binarization

Convert grayscale to binary (ridges white, valleys black).

| Method | Key params | Notes |
|---|---|---|
| `fixed` | thresh | Simple global threshold |
| `otsu` | — | Automatic global threshold (histogram valley) |
| `li` | — | Li minimum cross-entropy |
| `triangle` | — | Triangle method (skewed histograms) |
| `mean` | — | Mean pixel value as threshold |
| `adaptive` / `adaptive_gaussian` | block_size, C | Local Gaussian-weighted |
| `adaptive_mean` | block_size, C | Local mean-weighted |
| `adaptive_gaussian_blurred` | blur_ksize, block_size, C | Pre-blurred adaptive (most robust) |
| `sauvola` | window_size, k | Sauvola — accounts for local std dev |
| `niblack` | window_size, k | Niblack — similar, less noise tolerant |


In [ ]:
# ── Shared params ────────────────────────────────────────────────────────────
block  = 21     # window size for adaptive / sauvola / niblack
C      = 2      # constant subtracted from mean
blur   = 3      # pre-blur for *_blurred variant
thresh = 127    # fixed threshold
k_sauv = 0.2   # Sauvola k (sensitivity to std dev)
k_nibl = -0.2  # Niblack k (negative → more pixels as foreground)

src = ENH

results = [
    ("Input",          src),
    ("fixed",          fpbin.fixed_binarize(src, thresh=thresh)),
    ("otsu",           fpbin.otsu_binarize(src)),
    ("li",             fpbin.li_binarize(src)),
    ("triangle",       fpbin.triangle_binarize(src)),
    ("mean",           fpbin.mean_binarize(src)),
    ("adaptive",       fpbin.adaptive_binarize(src, block_size=block, C=C)),
    ("adaptive_mean",  fpbin.adaptive_mean_binarize(src, block_size=block, C=C)),
    ("adapt_gauss",    fpbin.adaptive_gaussian_binarize(src, block_size=block, C=C)),
    ("adapt_blur",     fpbin.adaptive_gaussian_binarize_blurred(
                           src, blur_ksize=blur, block_size=block, C=C)),
    (f"sauvola  k={k_sauv}", fpbin.sauvola_binarize(src, window_size=block, k=k_sauv)),
    (f"niblack  k={k_nibl}", fpbin.niblack_binarize(src, window_size=block, k=k_nibl)),
]

show(results, cols=4)

# ── Choose one and apply the mask ─────────────────────────────────────────────
BIN_RAW = fpbin.adaptive_gaussian_binarize_blurred(src, blur_ksize=blur,
                                                    block_size=block, C=C)
BIN = fpbin.apply_mask_to_binary(BIN_RAW, MASK)

# Auto-correct polarity: ridges should be WHITE
BIN = fpbin.invert_binary_if_needed(BIN, want_ridges_white=True, mask=MASK)

show([("Chosen binary + mask", BIN)], cols=1)
print("BIN ready.")


---
## Phase 6 — Morphology

Clean up the binary image: remove noise, fill gaps, smooth ridge boundaries.

| Function | Effect |
|---|---|
| `morph_open(k)` | Erosion → dilation — removes speckle |
| `morph_close(k)` | Dilation → erosion — fills small gaps |
| `morph_erode(k)` | Shrink white regions |
| `morph_dilate(k)` | Expand white regions |
| `fill_holes` | Fill enclosed black regions inside ridges |
| `remove_small_components(min_area)` | Delete isolated blobs |
| `bridge_gaps(k)` | Close small breaks in ridges |
| `despeckle_binary(min_area)` | Remove tiny noise blobs |
| `apply_morphology(close_k, open_k, min_area, fill)` | All-in-one convenience |


In [ ]:
k_open  = 3
k_close = 3
k_erode = 2
k_dilate= 2
min_area = 30

src = BIN

show([
    ("Binary input",   src),
    (f"open  k={k_open}",    fpmorph.morph_open(src, ksize=k_open)),
    (f"close  k={k_close}",  fpmorph.morph_close(src, ksize=k_close)),
    (f"erode  k={k_erode}",  fpmorph.morph_erode(src, ksize=k_erode)),
    (f"dilate  k={k_dilate}",fpmorph.morph_dilate(src, ksize=k_dilate)),
    ("fill_holes",           fpmorph.fill_holes(src)),
    (f"remove_small  a={min_area}", fpmorph.remove_small_components(src, min_area=min_area)),
    (f"bridge_gaps  k={k_close}",   fpmorph.bridge_gaps(src, ksize=k_close)),
    (f"despeckle  a={min_area}",    fpmorph.despeckle_binary(src, min_area=min_area)),
    ("all-in-one",           fpmorph.apply_morphology(src, close_k=k_close,
                                 open_k=k_open, min_area=min_area, fill=True)),
])

# Choose cleaned binary
MORPH = fpmorph.apply_morphology(BIN, close_k=k_close, open_k=k_open,
                                  min_area=min_area, fill=True)
print("MORPH ready.")


---
## Phase 7 — Skeletonisation

Thin ridges to a 1-pixel-wide medial axis.

| Function | Notes |
|---|---|
| `get_skeleton(use_skimage=True)` | skimage thinning (higher quality) |
| `get_skeleton(use_skimage=False)` | OpenCV morphological thinning (faster) |
| `remove_short_spurs(spur_length)` | Delete short branches hanging off junctions |
| `remove_small_skeleton_components(min_size)` | Remove isolated skeleton fragments |
| `prune_skeleton_topology(spur_length, min_component_size, n_passes)` | Full multi-pass prune |


In [ ]:
spur_len  = 8
min_comp  = 12
n_passes  = 3

src = MORPH

skel_sk = fpskel.get_skeleton(src, use_skimage=True)
skel_cv = fpskel.get_skeleton(src, use_skimage=False)

skel_pruned = fpskel.prune_skeleton_topology(skel_sk,
    spur_length=spur_len, min_component_size=min_comp, n_passes=n_passes)

skel_no_spurs = fpskel.remove_short_spurs(skel_sk, spur_length=spur_len)
skel_no_small = fpskel.remove_small_skeleton_components(skel_sk, min_size=min_comp)

def overlay(img, skel, color=(0, 220, 0)):
    rgb = cv2.cvtColor(normalize(img), cv2.COLOR_GRAY2BGR)
    rgb[skel > 0] = color
    return rgb

show([
    ("Binary input",             src),
    ("Skeleton (skimage)",        skel_sk),
    ("Skeleton (OpenCV)",         skel_cv),
    (f"Remove spurs  len={spur_len}", skel_no_spurs),
    (f"Remove small  min={min_comp}", skel_no_small),
    (f"Full prune  {n_passes} pass",  skel_pruned),
    ("Pruned overlay on binary",  overlay(src, skel_pruned)),
])

SKEL = skel_pruned
print("SKEL ready.")


---
## Phase 8 — Minutiae extraction

Extract and visualise ridge endings and bifurcations.

| Function | Purpose |
|---|---|
| `extract_minutiae_crossing_number` | Main extractor — crossing number algorithm |
| `finalize_minutiae` | Remove spurs and suppress duplicate detections |
| `draw_minutiae` | Render to image with legend |


In [ ]:
border_margin = 12    # ignore minutiae this close to ROI edge
cluster_dist  = 6     # merge minutiae closer than this (pixels)
spur_len_fin  = 15    # spur length to suppress in finalization
suppress_dist = 12    # suppression radius for duplicates

src = SKEL

raw  = fpmin.extract_minutiae_crossing_number(src,
           mask=MASK, border_margin=border_margin, cluster_dist=cluster_dist)
fin  = fpmin.finalize_minutiae(raw, src,
           max_spur_len=spur_len_fin, suppress_dist=suppress_dist)

vis_raw = fpmin.draw_minutiae(raw, background=src, show_arrows=True)
vis_fin = fpmin.draw_minutiae(fin, background=src, show_arrows=True)

n_end_r = sum(1 for m in raw if m["type"] == "ending")
n_bif_r = sum(1 for m in raw if m["type"] == "bifurcation")
n_end_f = sum(1 for m in fin if m["type"] == "ending")
n_bif_f = sum(1 for m in fin if m["type"] == "bifurcation")

print(f"Raw:       {len(raw)} minutiae  ({n_end_r} endings, {n_bif_r} bifurcations)")
print(f"Finalized: {len(fin)} minutiae  ({n_end_f} endings, {n_bif_f} bifurcations)")

show([
    (f"Raw  ({len(raw)})",       vis_raw),
    (f"Finalized  ({len(fin)})", vis_fin),
], figsize_per=5)

MINUTIAE = fin
print("MINUTIAE ready.")


---
## Phase 9 — Quality & singularities

### Quality score
`estimate_image_quality` — returns a 0–1 score based on frequency-domain clarity.
Higher = cleaner ridge structure.

### Singularity detection
Core points (Poincaré index = +0.5) and delta points (index = −0.5) encode the
fingerprint class (whorl, arch, loop).

| Function | Notes |
|---|---|
| `estimate_local_orientation` | Block-wise orientation angle map |
| `extract_singularities` | Core / delta from Poincaré index of orientation field |


In [ ]:
# Quality
quality = fpqual.estimate_image_quality(img_gray, mask=MASK)
print(f"Image quality score: {quality:.3f}  (0=poor, 1=excellent)")

# Orientation map
orient_block = 16
theta = fpenh.estimate_local_orientation(normalize(ENH), block=orient_block)

# Singularities
sings = fpsin.extract_singularities(theta, mask=MASK)
cores  = [s for s in sings if s["type"] == "core"]
deltas = [s for s in sings if s["type"] == "delta"]
print(f"Singularities: {len(cores)} core(s), {len(deltas)} delta(s)")

# Visualise orientation as HSV hue overlay
theta_norm = ((theta + np.pi / 2) / np.pi * 179).astype(np.uint8)
hsv = np.zeros((*theta_norm.shape, 3), dtype=np.uint8)
hsv[..., 0] = theta_norm
hsv[..., 1] = 200
hsv[..., 2] = np.where(MASK > 0, 200, 0).astype(np.uint8)
orient_rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

# Mark singularities on image
sing_vis = cv2.cvtColor(normalize(img_gray), cv2.COLOR_GRAY2BGR)
for s in sings:
    col = (0, 0, 255) if s["type"] == "core" else (255, 0, 0)
    cv2.circle(sing_vis, (s["x"], s["y"]), 8, col, 2)
    cv2.putText(sing_vis, s["type"][0].upper(),
                (s["x"] + 10, s["y"]), cv2.FONT_HERSHEY_SIMPLEX, 0.5, col, 1)

show([
    ("Orientation field (hue=angle)", orient_rgb),
    ("Singularities  (red=core, blue=delta)", sing_vis),
], figsize_per=5)


---
## Phase 10 — Scale calibration & print DPI

Use a known physical distance in the image to set the scale, then rescale for printing.

| Function | Purpose |
|---|---|
| `ScaleCalibrator` | Interactive widget — click two points on a known object |
| `rescale_to_print_dpi(img, px_per_mm, dpi)` | Rescale image to target print resolution |
| `print_size_mm(img, px_per_mm)` | Check physical print size at given resolution |
| `estimate_ridge_frequency` | Auto-estimate scale from ridge spacing (no reference needed) |
| `normalize_to_dpi` | Rescale so ridge spacing matches a target DPI |


In [ ]:
# ── Auto-scale from ridge frequency (no physical reference needed) ───────────
# Ridge spacing at 500 DPI is ~8-12 pixels (depends on scanner/camera).
# This estimates the current pixels-per-ridge-period and rescales.

src_for_freq = normalize(ENH)
try:
    ridge_freq_px = fpfreq.auto_detect_scale(src_for_freq, mask=MASK)
    print(f"Estimated ridge period: {1/ridge_freq_px:.1f} px/ridge")

    img_500dpi = fpfreq.normalize_to_dpi(src_for_freq,
                     current_freq=ridge_freq_px, target_dpi=500)
    print(f"Rescaled shape for 500 DPI: {img_500dpi.shape}")
    show([("Original", src_for_freq), ("Rescaled → 500 DPI equiv.", img_500dpi)])
except Exception as ex:
    print(f"Auto-scale failed: {ex}")

# ── Manual calibration widget (run this cell to open the UI) ─────────────────
if fpcal is not None:
    print("\nLaunch the scale calibrator:")
    print("  1. Run:  cal = fpcal.ScaleCalibrator(img_gray); cal.display()")
    print("  2. Click two points of KNOWN distance in mm")
    print("  3. Read cal.px_per_mm, then:")
    print("     rescaled = fpcal.rescale_to_print_dpi(valley_print, cal.px_per_mm, dpi=300)")
    print("     w_mm, h_mm = fpcal.print_size_mm(rescaled, cal.px_per_mm)")
else:
    print("calibration module not available")


---
## Phase 11 — Minutiae matching

Compare two fingerprints by aligning their minutiae sets and counting correspondences.

| Function | Returns |
|---|---|
| `align_minutiae(A, B)` | Best rigid transform + matched pairs |
| `match_score(A, B, pairs)` | Score 0–1 |
| `compare_fingerprints(img_a, min_a, img_b, min_b)` | All-in-one: align + score + visualise |


In [ ]:
# ── Load a second fingerprint (edit the path) ────────────────────────────────
IMAGE_PATH_B = PROJECT_ROOT / "images" / "photo" / "image4.jpeg"  # ← change me

img_b_bgr  = cv2.imread(str(IMAGE_PATH_B))
img_b_gray = cv2.cvtColor(img_b_bgr, cv2.COLOR_BGR2GRAY) if img_b_bgr is not None else img_gray

# Quick pipeline for image B (reuse same params as tuned above)
enh_b  = fpenh.apply_clahe(img_b_gray)
bin_b  = fpbin.adaptive_gaussian_binarize_blurred(enh_b)
bin_b  = fpbin.invert_binary_if_needed(bin_b)
morph_b= fpmorph.apply_morphology(bin_b)
skel_b = fpskel.prune_skeleton_topology(fpskel.get_skeleton(morph_b))
min_b  = fpmin.finalize_minutiae(
             fpmin.extract_minutiae_crossing_number(skel_b), skel_b)

# ── Compare ──────────────────────────────────────────────────────────────────
dist_tol  = 20.0   # px — matching distance tolerance
angle_tol = 0.35   # radians — matching angle tolerance

result_vis, matched_pairs, score = fpmat.compare_fingerprints(
    img_gray, MINUTIAE,
    img_b_gray, min_b,
    dist_tol=dist_tol, angle_tol=angle_tol,
    label_a="Image A", label_b="Image B",
)

print(f"Match score: {score:.3f}   ({len(matched_pairs)} matched pairs)")
plt.figure(figsize=(14, 6))
plt.imshow(cv2.cvtColor(result_vis, cv2.COLOR_BGR2RGB))
plt.title(f"Match score = {score:.3f}")
plt.axis("off"); plt.show()


---
## Full pipeline summary

The variables produced by each phase, available for downstream use:

| Variable | Content |
|---|---|
| `img_gray` | Raw grayscale input |
| `MASK` | ROI binary mask (255 inside) |
| `ILLUM` | Illumination-corrected image |
| `ENH` | Contrast-enhanced image |
| `BIN` | Binary (ridges=white, valleys=black) |
| `MORPH` | Morphologically cleaned binary |
| `SKEL` | Skeleton (1-px ridges) |
| `MINUTIAE` | List of minutia dicts with x, y, type, angle |
